# Q. Bayesian Estimation of a User Ability Parameter from Item Responses

An online learning platform presents a user with a sequence of $n$ multiple-choice questions **one at a time**. Each question is either answered correctly or incorrectly, allowing the platform to update its estimate of the user's ability dynamically after every response.

Let $Y_i$ denote the user's response to the $i$-th item encountered:

$$Y_i=
\begin{cases}
1, & \text{if the user answers item } i \text{ correctly},\\
0, & \text{if the user answers item } i \text{ incorrectly}.
\end{cases}$$

The platform assumes that the probability of a correct response is governed by a two-parameter logistic (2PL) item response model. Specifically, conditional on the user's latent ability parameter $\Theta=\theta$, the response probability for item $i$ is:

$$P(Y_i=1\mid \Theta=\theta)=p_i(\theta)=\frac{1}{1+e^{-a_i(\theta-b_i)}},$$

where $a_i>0$ is the known discrimination parameter, and $b_i$ is the known difficulty parameter of item $i$.

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running vector of observed responses** up to the current step $k$ (where $1 \le k \le n$).

Before observing any responses, the platform initializes the user's latent ability estimate with a standard normal prior distribution:

$$f_{\Theta}^{(0)}(\theta) = \frac{1}{\sqrt{2\pi}} \exp\left(-\frac{\theta^2}{2}\right) \quad \text{implying} \quad \Theta \sim \mathscr{N}(0,1).$$

As the user progresses, the posterior distribution at step $k-1$ serves as the prior distribution for step $k$.

---

### Tasks

1. **Visualizing the Mechanics:** Plot $P(Y_i=1\mid \Theta=\theta)$ vs $\theta$ using Plotly for two distinct values of $a_i$, where one of those $a_i$ values is paired with three different difficulty values of $b_i$. Interpret how moving $b_i$ shifts the curve horizontally.
2. **Sequential Likelihood Contribution:** Write down the likelihood contribution $L(y_k \mid \theta)$ of a *single* new response $y_k$ at step $k$, given the latent ability $\theta$. Then, write down the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.
3. **Mathematical Formulation of the Running Update:** Write down the recursive relationship for the posterior density at step $k$, denoted $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$, up to a proportionality constant, using the prior state $f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$ and the new observation $y_k$.
4. **Dynamic Shifting:** Explain how a correct answer ($y_k = 1$) to a highly difficult item (large $b_k$) mathematically shifts the peak of the running posterior density distribution relative to the previous step.
5. **Tracking Certainty and Sharpness:** Explain how the discrimination parameter $a_k$ of the current item alters the variance (or "sharpness") of the distribution during a running update. What happens when $a_k$ is very large versus very small?
6. **Numerical Implementation of a Running Grid:** Describe a algorithmic approach to numerically approximate and maintain this running posterior density function on a fixed grid of $\theta$-values. Explicitly state how you would perform the sequential normalization step computationally after an item is answered.


7. **Evaluating Convergence over the Timeline:** Suppose the user's true, hidden latent ability is $\theta_{\text{true}} = 0.75$. Write a Python script that extends your previous grid simulation to track the performance of the running estimators over a sequence of $n = 20$ items.
* **Simulate Responses:** Dynamically generate the user's responses $y_k \in \{0, 1\}$ at each step by comparing a random draw from a Uniform distribution $U(0,1)$ against the true response probability $p_k(\theta_{\text{true}})$. Give each item a random difficulty $b_k \sim \mathscr{N}(0, 1)$ and a random discrimination $a_k \sim \text{Uniform}(0.5, 2.0)$.
* **Track Estimators:** At each step $k$, calculate and store the running Posterior Mean ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$) and the running Maximum A Posteriori ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$) estimate.
* **Visualize:** Use Plotly to create a single line chart showing the progression of both estimators from step $0$ to $20$. Add a static horizontal reference line at $y = 0.75$ representing $\theta_{\text{true}}$.
* **Analysis:** Briefly explain how the distance between your estimators and $\theta_{\text{true}}$ changes as $k$ increases, and interpret what this implies about the platform's confidence in its measurement.


1.

In [1]:
import numpy as np
import plotly.graph_objects as go

# Define theta range
theta = np.linspace(-4, 4, 200)

# 2PL model function
def p_correct(theta, a, b):
    return 1 / (1 + np.exp(-a * (theta - b)))

# Define item parameters to compare
items = [
    {"a": 1.0, "b": 0.0, "name": "Standard (a=1, b=0)", "dash": "solid"},
    {"a": 2.0, "b": -1.5, "name": "Easy, High Disc (a=2, b=-1.5)", "dash": "dash"},
    {"a": 2.0, "b": 0.0, "name": "Med, High Disc (a=2, b=0)", "dash": "dash"},
    {"a": 2.0, "b": 1.5, "name": "Hard, High Disc (a=2, b=1.5)", "dash": "dash"}
]

fig = go.Figure()

for item in items:
    prob = p_correct(theta, item["a"], item["b"])
    fig.add_trace(go.Scatter(
        x=theta, y=prob,
        mode='lines',
        name=item["name"],
        line=dict(dash=item["dash"], width=3)
    ))

fig.update_layout(
    title="Item Characteristic Curves (2PL Model)",
    xaxis_title="Latent Ability (theta)",
    yaxis_title="Probability of Correct Response P(Y=1 | theta)",
    legend=dict(x=0.05, y=0.95),
    template="plotly_white"
)

fig.show()

Moving $b_i$ shifts the logistic curve horizontally. The difficulty parameter $b_i$ is explicitly the point on the $\theta$ scale where the user has a exactly a 50% chance of answering correctly.

- A positive $b_i$ shifts the curve to the right, meaning the user needs a higher ability ($\theta$) to achieve the same probability of success (a harder item).

- A negative $b_i$ shifts the curve to the left (an easier item).

2. Because each response is a binary outcome (correct or incorrect), it follows a Bernoulli distribution. The likelihood contribution of a single new response $y_k$ given latent ability $\theta$ is:$$L(y_k \mid \theta) = [p_k(\theta)]^{y_k} [1 - p_k(\theta)]^{1 - y_k}$$Where $p_k(\theta) = \frac{1}{1+e^{-a_k(\theta - b_k)}}$.Assuming local independence (the user's response to one item doesn't affect another, conditional on their ability), the joint likelihood function for the running history vector $y^{(k)}$ is the product of the individual likelihoods:$$L(y^{(k)} \mid \theta) = \prod_{i=1}^{k} [p_i(\theta)]^{y_i} [1 - p_i(\theta)]^{1 - y_i}$$

3. According to Bayes' theorem, the posterior is proportional to the prior multiplied by the likelihood. Because we update sequentially, yesterday's posterior becomes today's prior.

The recursive relationship for the posterior density at step $k$ is:$$f_{\Theta \mid Y^{(k)}}(\theta \mid y^{(k)}) \propto f_{\Theta \mid Y^{(k-1)}}(\theta \mid y^{(k-1)}) \times \left( [p_k(\theta)]^{y_k} [1 - p_k(\theta)]^{1 - y_k} \right)$$

4. If a user gets a highly difficult item correct ($y_k=1$ and $b_k \gg 0$), the likelihood function $L(y_k \mid \theta)$ will evaluate to near zero for lower values of $\theta$ and approach 1 only for very high values of $\theta$.

When we multiply the prior distribution by this likelihood, we heavily penalize the probability mass on the left side of the distribution. This mathematical suppression of low $\theta$ values forces the peak of the new posterior density (the mode) to shift drastically to the right, rapidly adjusting the platform's belief upward regarding the user's ability.

5. The discrimination parameter $a_k$ controls the slope (or steepness) of the logistic curve at $b_k$. It essentially dictates how much information the item provides.

-Large $a_k$ (High Discrimination): The likelihood function transitions from near 0 to near 1 very abruptly. When multiplied against the prior, it aggressively cuts off the distribution on one side. This causes a massive reduction in the variance of the posterior distribution, making the resulting density much "sharper" or narrower. The platform becomes highly certain, very quickly.

-Small $a_k$ (Low Discrimination): The logistic curve is flat, and the likelihood function hovers closer to 0.5 across a wide range of $\theta$. Multiplying the prior by this flat likelihood provides very little new information, meaning the posterior variance remains largely unchanged.

6. Because the 2PL model paired with a normal prior does not result in a closed-form conjugate posterior, we must approximate it. The simplest and most robust way in 1D is a grid approximation:

- Initialization: Define a discrete grid of $\theta$ values (e.g., $-4.0$ to $4.0$ in steps of $0.01$). Evaluate the standard normal PDF across this grid to form the initial prior array

- Update Step: When a new response $y_k$ arrives, calculate the likelihood array for that specific item across the entire $\theta$ grid.

- Multiplication: Perform element-wise multiplication of the current prior array and the likelihood array.

- Sequential Normalization: To turn the unnormalized array back into a true probability density function (PDF) that integrates to 1, sum all the elements in the array and multiply by the grid step size ($\Delta\theta$). Divide the unnormalized array by this scalar value.

7.

In [2]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import norm

# 1. Simulation Setup
np.random.seed(42)
n_items = 20
theta_true = 0.75

# Grid setup
theta_min, theta_max, n_grid = -4.0, 4.0, 801
theta_grid = np.linspace(theta_min, theta_max, n_grid)
d_theta = theta_grid[1] - theta_grid[0]

# Initialize Prior
posterior_density = norm.pdf(theta_grid, loc=0, scale=1)

# Arrays to track history
map_estimates = []
mean_estimates = []

# 2. Sequential Simulation and Updating
for k in range(n_items):
    # Generate random item parameters
    a_k = np.random.uniform(0.5, 2.0)
    b_k = np.random.normal(0, 1)

    # Simulate user response
    p_true = 1 / (1 + np.exp(-a_k * (theta_true - b_k)))
    y_k = 1 if np.random.uniform(0, 1) < p_true else 0

    # Calculate Likelihood on grid
    p_grid = 1 / (1 + np.exp(-a_k * (theta_grid - b_k)))
    likelihood = p_grid if y_k == 1 else (1 - p_grid)

    # Bayesian Update: Prior * Likelihood
    posterior_density = posterior_density * likelihood

    # Sequential Normalization
    area = np.sum(posterior_density) * d_theta
    posterior_density = posterior_density / area

    # Calculate Estimators
    current_map = theta_grid[np.argmax(posterior_density)]
    current_mean = np.sum(theta_grid * posterior_density) * d_theta

    map_estimates.append(current_map)
    mean_estimates.append(current_mean)

# 3. Visualization
steps = np.arange(1, n_items + 1)

fig = go.Figure()

fig.add_trace(go.Scatter(x=steps, y=mean_estimates, mode='lines+markers', name='Posterior Mean'))
fig.add_trace(go.Scatter(x=steps, y=map_estimates, mode='lines+markers', name='MAP Estimate'))

# Add true theta reference line
fig.add_hline(y=theta_true, line_dash="dash", line_color="red",
              annotation_text="True Theta (0.75)", annotation_position="bottom right")

fig.update_layout(
    title="Convergence of Bayesian Ability Estimators over 20 Items",
    xaxis_title="Item Number (k)",
    yaxis_title="Estimated Latent Ability (theta)",
    template="plotly_white",
    yaxis=dict(range=[-2, 2])
)

fig.show()

As $k$ increases, the distance between both estimators (MAP and Posterior Mean) and $\theta_{true}$ generally shrinks. In the first few items, the estimators will jump erratically because a single response carries massive weight relative to the uninformative prior. As the history vector $y^{(k)}$ grows, the accumulated evidence overwhelms single data points, and the distribution's variance narrows around the true parameter. This shrinking variance implies the platform is gaining confidence—it takes significantly more evidence (or highly surprising incorrect answers to easy questions) to shift the estimate later in the test than it does at the beginning.

# Q. Bayesian Tracking of Click-Through Rates (CTR) via Conjugate Beta-Binomial Updates

An e-commerce platform wants to optimize its recommendation engine by dynamically estimating the click-through rate (CTR) of a newly launched advertisement. Since user traffic arrives continuously, the platform updates its belief about the advertisement's performance **one impression at a time** rather than waiting for large batch updates.

Let $\Theta = \theta$ represent the true, hidden conversion rate (probability of a click) of the advertisement, where $\theta \in [0, 1]$.

Let $Y_k$ denote a single user's interaction with the advertisement at time step $k$:

$$Y_k =
\begin{cases}
1, & \text{if the user clicks the advertisement}, \\
0, & \text{if the user does not click the advertisement}.
\end{cases}$$

The platform assumes that conditional on the true conversion rate $\Theta = \theta$, each user interaction is an independent Bernoulli trial:

$$P(Y_k = 1 \mid \Theta = \theta) = \theta$$

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running vector of observed user interactions** up to the current impression step $k$ (where $1 \le k \le n$).

Before observing any data, the platform assigns a flexible **Beta distribution** as the initial prior over the unknown parameter $\Theta$:

$$f_{\Theta}^{(0)}(\theta) = \frac{1}{\mathrm{B}(\alpha_0, \beta_0)} \theta^{\alpha_0 - 1} (1 - \theta)^{\beta_0 - 1} \quad \text{implying} \quad \Theta \sim \text{Beta}(\alpha_0, \beta_0)$$

where $\mathrm{B}(\cdot, \cdot)$ is the Beta function acting as the normalizing constant. Under a sequential framework, the posterior distribution at step $k-1$ serves directly as the prior distribution for step $k$.

---

**Tasks**

**1. Structural Probability and Properties**
Plot the probability density function (PDF) of a $\text{Beta}(\alpha, \beta)$ distribution using Plotly for three distinct parameter pairs:

* Uninformative state: $(\alpha=1, \beta=1)$
* Right-skewed state: $(\alpha=2, \beta=8)$
* Left-skewed state: $(\alpha=8, \beta=2)$

Interpret how changing the balance between $\alpha$ and $\beta$ shifts the center of mass of the density function over the domain $[0, 1]$.

**2. Sequential Likelihood and Joint History**

Write down the mathematical likelihood contribution $L(y_k \mid \theta)$ of a *single* isolated response $y_k$ at step $k$, given the click probability $\theta$. Following this, express the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.

**3. Closed-Form Analytical Updates (Conjugacy)**

Using Bayes' Theorem, derive the recursive algebraic relationship for the posterior density at step $k$, denoted as $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$. Prove analytically that the posterior remains in the Beta family (**Beta-Binomial Conjugacy**) by explicitly writing down the closed-form update parameters $\alpha_k$ and $\beta_k$ as simple arithmetic updates of $\alpha_{k-1}$, $\beta_{k-1}$, and $y_k$. Also compute the **Posterior Mean** of the latent parameter $\Theta$ at time step $k$ (i.e. $\mathbb{E}[\Theta \mid \mathbf{Y}^{(k)}=\mathbf{y}^{(k)}]$).


**4. Dynamic Shifting Mechanics**

Explain how an observed click ($y_k = 1$) vs. a non-click ($y_k = 0$) shifts the peak of the running density distribution mathematically. Contrast this analytical framework against non-conjugate setups (such as the 2PL IRT model) where numerical grid integration is strictly required.

**5. Running Point Estimators**

State the exact closed-form equations used to evaluate the following point estimates at step $k$ directly from the updated shape parameters $\alpha_k$ and $\beta_k$:

* **Running Posterior Mean** ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$)
* **Running Maximum A Posteriori** ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$)

**6. Performance Tracking and Convergence Analysis**

Suppose the advertisement's true, hidden click-through rate is $\theta_{\text{true}} = 0.35$. Write a Python script to track the performance of your closed-form sequential estimators over a timeline of $n = 100$ impressions:

* **Initialize State:** Set the base prior parameters to $\alpha_0 = 1, \beta_0 = 1$ (representing uniform initial uncertainty).
* **Simulate Responses:** Dynamically generate user responses $y_k \in \{0, 1\}$ at each step by comparing a random draw from a Uniform distribution $U(0,1)$ against $\theta_{\text{true}}$.
* **Track Estimators:** Loop through each step, update $\alpha_k$ and $\beta_k$ analytically, and store the computed values for $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$.
* **Visualize:** Use Plotly to create a single line chart showing the progression of both estimators from step $0$ to $100$. Add a static horizontal reference line at $y = 0.35$ representing $\theta_{\text{true}}$.
* **Analysis:** Explain how the distance between your estimators and $\theta_{\text{true}}$ responds as the sampling size $k$ approaches $100$. What does this imply about the accumulation of evidence over time relative to the choice of the initial prior?

1.

In [3]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta

theta = np.linspace(0, 1, 500)

priors = [
    {"alpha": 1, "beta": 1, "name": "Uninformative (1, 1)", "dash": "solid"},
    {"alpha": 2, "beta": 8, "name": "Right-Skewed (2, 8)", "dash": "dash"},
    {"alpha": 8, "beta": 2, "name": "Left-Skewed (8, 2)", "dash": "dash"}
]

fig = go.Figure()

for p in priors:
    pdf = beta.pdf(theta, p["alpha"], p["beta"])
    fig.add_trace(go.Scatter(
        x=theta, y=pdf,
        mode='lines',
        name=p["name"],
        line=dict(dash=p["dash"], width=3)
    ))

fig.update_layout(
    title="Beta Distribution Prior Shapes",
    xaxis_title="Click-Through Rate (theta)",
    yaxis_title="Density",
    template="plotly_white"
)

fig.show()

- $\text{Beta}(1, 1)$: This is mathematically equivalent to a Uniform distribution on $[0,1]$. It implies complete ignorance—every possible CTR is equally likely.

- $\text{Beta}(2, 8)$: The center of mass shifts heavily toward $0$. This represents a pessimistic prior belief where we expect the CTR to be low (centered around $\frac{2}{2+8} = 20\%$).

- $\text{Beta}(8, 2)$: The center of mass shifts toward $1$. This is a highly optimistic prior, implying we expect an $80\%$ CTR.

2. Because each impression results in a binary outcome (click or no click), it is a Bernoulli trial. The likelihood of a single observation $y_k$ given $\theta$ is:$$L(y_k \mid \theta) = \theta^{y_k} (1 - \theta)^{1 - y_k}$$Assuming independent impressions, the joint likelihood function for the running history vector $y^{(k)}$ is the product of individual likelihoods:$$L(y^{(k)} \mid \theta) = \prod_{i=1}^{k} \theta^{y_i} (1 - \theta)^{1 - y_i}$$

3. Conjugacy means that if the prior is a Beta distribution and the likelihood is Binomial/Bernoulli, the resulting posterior is guaranteed to also be a Beta distribution.

By Bayes' Theorem, the posterior is proportional to the prior times the likelihood. At step $k$, using the prior from step $k-1$:$$f_{\Theta \mid Y^{(k)}}(\theta \mid y^{(k)}) \propto f_{\Theta \mid Y^{(k-1)}}(\theta \mid y^{(k-1)}) \times L(y_k \mid \theta)$$Substitute the Beta PDF (ignoring normalizing constants) and the Bernoulli likelihood:$$f_{\Theta \mid Y^{(k)}}(\theta \mid y^{(k)}) \propto \left[ \theta^{\alpha_{k-1}-1} (1-\theta)^{\beta_{k-1}-1} \right] \times \left[ \theta^{y_k} (1-\theta)^{1-y_k} \right]$$When multiplying terms with the same base, you add the exponents:$$f_{\Theta \mid Y^{(k)}}(\theta \mid y^{(k)}) \propto \theta^{(\alpha_{k-1} + y_k) - 1} (1-\theta)^{(\beta_{k-1} + 1 - y_k) - 1}$$This resulting expression matches the exact functional form of a new Beta distribution, proving conjugacy. We can directly read off the updated parameters:$$\alpha_k = \alpha_{k-1} + y_k$$$$\beta_k = \beta_{k-1} + (1 - y_k)$$The Posterior Mean at time step $k$ is simply the expectation of this new Beta distribution:$$E[\Theta \mid Y^{(k)} = y^{(k)}] = \frac{\alpha_k}{\alpha_k + \beta_k}$$

4. An observed click ($y_k=1$) increases $\alpha_k$ by 1, shifting the distribution peak to the right. A non-click ($y_k=0$) increases $\beta_k$ by 1, shifting the peak to the left. Because the prior and posterior are in the same algebraic family, the update is a simple arithmetic addition. This sharply contrasts with non-conjugate setups (like the 2PL IRT model) which lack closed-form algebraic solutions and mandate computationally intensive numerical grid integration.

5. Because we know the exact parameters $\alpha_k$ and $\beta_k$ at any time $k$, we can query point estimates instantaneously without integration:Running Posterior Mean:$$\hat{\theta}^{(k)}_{Bayes} = \frac{\alpha_k}{\alpha_k + \beta_k}$$Running Maximum A Posteriori (MAP) (The Mode):$$\hat{\theta}^{(k)}_{MAP} = \frac{\alpha_k - 1}{\alpha_k + \beta_k - 2}$$The MAP is technically only defined when $\alpha_k > 1$ and $\beta_k > 1$

6.

In [1]:
import numpy as np
import plotly.graph_objects as go

# 1. Simulation Setup
np.random.seed(101)
n_impressions = 100
theta_true = 0.35

# Initialize State (Uninformative Prior)
alpha_k = 1.0
beta_k = 1.0

bayes_estimates = []
map_estimates = []

# 2. Sequential Simulation and Updating
for k in range(n_impressions):
    # Simulate user interaction
    y_k = 1 if np.random.uniform() < theta_true else 0

    # Analytical Conjugate Update
    alpha_k += y_k
    beta_k += (1 - y_k)

    # Compute Estimators
    current_mean = alpha_k / (alpha_k + beta_k)

    # Handle MAP boundaries (undefined for alpha=1, beta=1)
    if (alpha_k + beta_k - 2) > 0:
        current_map = (alpha_k - 1) / (alpha_k + beta_k - 2)
        # Cap to [0,1] bounds
        current_map = max(0, min(1, current_map))
    else:
        current_map = 0.5

    bayes_estimates.append(current_mean)
    map_estimates.append(current_map)

# 3. Visualization
steps = np.arange(1, n_impressions + 1)

fig = go.Figure()

fig.add_trace(go.Scatter(x=steps, y=bayes_estimates, mode='lines', name='Posterior Mean'))
fig.add_trace(go.Scatter(x=steps, y=map_estimates, mode='lines', name='MAP Estimate'))

# True theta reference line
fig.add_hline(y=theta_true, line_dash="dash", line_color="red",
              annotation_text="True CTR (0.35)", annotation_position="bottom right")

fig.update_layout(
    title="Convergence of CTR Estimators (Beta-Binomial Update)",
    xaxis_title="Impression Number (k)",
    yaxis_title="Estimated CTR",
    template="plotly_white",
    yaxis=dict(range=[0, 1])
)

fig.show()

# Q Bayesian Estimations for Structural Health Monitoring via Bounded Grid Updates

In aerospace and civil engineering, Structural Health Monitoring (SHM) is critical for detecting damage before a catastrophic failure occurs. Consider an aircraft wing or a bridge girder equipped with specialized vibration sensors. Over time, environmental fatigue or dynamic impacts can cause micro-fractures, resulting in a reduction of the component's mechanical stiffness.

Let $\Theta = \theta$ represent the structural **remaining stiffness efficiency factor**, where $\theta$ is physically bounded to the interval:

$$\theta \in (0, 1]$$

* $\theta = 1.0$ indicates a perfectly pristine, undamaged structural component.
* $\theta \to 0$ signifies critical degradation or severe structural cracking.

Let $K_{\text{nominal}}$ be the known, baseline stiffness of the structural component when it is entirely healthy. At each sequential inspection time step $k$ (where $k = 1, 2, \dots, n$), a sensor collects a noisy experimental stiffness measurement $y_k$.

Engineers model the degradation physics via a non-linear relationship with multiplicative log-normal measurement noise to prevent non-physical negative values:

$$y_k = \theta \cdot K_{\text{nominal}} \cdot e^{\epsilon_k}, \qquad \epsilon_k \sim \mathscr{N}(0, \sigma^2)$$

where $\sigma$ is the standard deviation of the sensor noise in log-space.

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running history vector of observed sensor readings** up to the current inspection milestone. Before deploying the sensors, engineers utilize an initial prior distribution $f_{\Theta}^{(0)}(\theta)$ over the domain $(0, 1]$ based on historical manufacturing specifications. As the sensor stream arrives, the posterior distribution calculated at step $k-1$ serves directly as the prior distribution for step $k$.

---

### **Tasks**

#### **1. Prior Belief Boundaries**

Before data collection begins, engineers assume the component is highly likely to be healthy, modeling this using a bounded Beta distribution as the initial prior: $\Theta \sim \text{Beta}(8, 1.5)$.

* Plot this initial prior density function using Plotly over the restricted physical domain $\theta \in [0.01, 1.0]$.
* Calculate the expected prior stiffness efficiency $\mathbb{E}[\Theta^{(0)}]$ analytically. Explain why this specific distribution serves as an appropriate initial prior for an engineering component assumed to be healthy.

#### **2. Structural Likelihood Formulation**

Using the change of variables or properties of the log-normal distribution, write down the mathematical likelihood contribution $L(y_k \mid \theta)$ of a *single* continuous sensor measurement $y_k$ at inspection step $k$, given the true stiffness factor $\theta$. Following this, write down the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.

#### **3. Mathematical Formulation of the Non-Conjugate Grid Update**

Explain why an exact closed-form analytical solution for the posterior density $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$ does not exist when combining a Beta prior with this log-normal structural likelihood. Write down the recursive relationship for the posterior density at step $k$ up to a proportionality constant.

#### **4. Running Point Estimates**

Because a closed-form formula is unavailable, we must define point estimators through numerical integration. Write down the definite integral equations over the bounded domain $(0, 1]$ required to compute:

* The **Running Posterior Mean** ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$)
* The **Running Maximum A Posteriori** ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$)

#### **5. Algorithmic Grid Approximation and Normalization**

Describe the step-by-step numerical procedure to maintain this distribution on a discrete grid of $\theta$-values. Explicitly state how you would handle the boundary limits computationally and how you would perform the sequential normalization step using the trapezoidal rule after a new sensor reading $y_k$ is observed.

#### **6. Performance Tracking and Degradation Convergence Analysis**

Suppose an impact occurs, and the true, hidden remaining stiffness drops to $\theta_{\text{true}} = 0.68$. Write a Python script using Plotly to simulate an engineered monitoring timeline across $n = 15$ continuous sensor measurements ($K_{\text{nominal}} = 50.0 \text{ kN/mm}$, $\sigma = 0.15$):

* **Simulate Sensor Stream:** Programmatically generate noisy sensor readings $y_k$ by drawing random values from the underlying log-normal physics model centered at $\theta_{\text{true}}$.
* **Track Estimators:** Loop sequentially through each step. At each step, update the unnormalized grid, normalize it via `np.trapezoid`, and compute both $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$.
* **Visualize Curves & Timeline:** Generate two plots:
1. A plot showing the progression of the full posterior density curves at milestones $k \in \{0, 1, 2, 5, 10, 15\}$.
2. A line chart tracking the convergence of both $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$ from step $0$ to $15$ against a horizontal reference line at $\theta_{\text{true}} = 0.68$.


* **Analysis:** Evaluate the behavior of the distribution. How many sensor readings did it take for the system to overcome the initially optimistic "healthy" prior and confidently isolate the 68% damage state? What does the narrowing of the density curves imply about structural safety thresholds?

1. The initial prior is $\Theta \sim \text{Beta}(8, 1.5)$ bounded over $[0.01, 1.0]$.
Expected prior stiffness: $\mathbb{E}[\Theta^{(0)}] = \frac{8}{8 + 1.5} = \frac{8}{9.5} \approx 0.842$.
This serves as an appropriate initial prior because engineers assume the newly manufactured component is mostly healthy (peak near 1.0) but naturally allow for slight manufacturing imperfections, hence the slight leftward spread rather than a strict delta function at 1.0.

2. Because $y_k = \theta \cdot K_{\text{nominal}} \cdot e^{\epsilon_k}$ with $\epsilon_k \sim \mathscr{N}(0, \sigma^2)$, the measurement follows a Log-normal distribution:
  $$L(y_k \mid \theta) = \frac{1}{y_k \sigma \sqrt{2\pi}} \exp\left( - \frac{(\ln y_k - \ln(\theta K_{\text{nominal}}))^2}{2\sigma^2} \right)$$The joint likelihood is:$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{j=1}^k \frac{1}{y_j \sigma \sqrt{2\pi}} \exp\left( - \frac{(\ln y_j - \ln(\theta K_{\text{nominal}}))^2}{2\sigma^2} \right)$$

  3. There is no algebraic property allowing a Beta distribution multiplied by a Log-normal likelihood to yield another recognized parameterized probability distribution. Hence, it is non-conjugate. The recursive relationship is strictly numerical:$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)}) \times \exp\left( - \frac{(\ln y_k - \ln(\theta K_{\text{nominal}}))^2}{2\sigma^2} \right)$$

  4. Because there is no closed form, point estimators are calculated via definite integration over the bounded domain $(0, 1]$:
  
  Running Posterior Mean: $\widehat{\theta}_{\mathrm{Bayes}}^{(k)} = \int_{0.01}^{1.0} \theta \cdot f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \, d\theta$
  
  Running MAP: $\widehat{\theta}_{\mathrm{MAP}}^{(k)} = \operatorname{arg\,max}_{\theta \in (0.01, 1.0]} f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$

  5.
  
  - Discretize the domain $[0.01, 1.0]$ into a fixed grid of $N$ points.
  
  - Initialize the density using the Beta(8, 1.5) PDF.
  
  - For each sensor reading $y_k$, evaluate the log-normal likelihood function across the entire grid.
  
  - Multiply the prior array by the likelihood array.
  
  - Use np.trapezoid on the grid array and the unnormalized density array to compute the area. Divide the density array by this area to enforce boundary limits and sequence normalization.

In [2]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta, lognorm

# Parameters
K_nom = 50.0
sigma = 0.15
theta_true = 0.68
n_steps = 15

# Grid setup (Boundary handling)
theta_grid = np.linspace(0.01, 1.0, 1000)
prior = beta.pdf(theta_grid, 8, 1.5)

# --- Task 1: Plot Initial Prior ---
fig5 = go.Figure()
fig5.add_trace(go.Scatter(x=theta_grid, y=prior, name="Prior Beta(8, 1.5)"))
fig5.update_layout(title="Initial Structural Health Prior", xaxis_title="Theta (Stiffness)", yaxis_title="Density")
fig5.show()

# --- Task 6: Simulation ---
np.random.seed(42)
posteriors = {0: prior.copy()}
mean_est = [np.trapezoid(theta_grid * prior, theta_grid)]
map_est = [theta_grid[np.argmax(prior)]]

posterior = prior.copy()

for k in range(1, n_steps + 1):
    # Simulate log-normal sensor reading
    y_k = theta_true * K_nom * np.exp(np.random.normal(0, sigma))

    # Compute likelihood
    likelihood = (1 / (y_k * sigma * np.sqrt(2 * np.pi))) * np.exp(- (np.log(y_k) - np.log(theta_grid * K_nom))**2 / (2 * sigma**2))

    # Update and Normalize
    unnormalized = posterior * likelihood
    posterior = unnormalized / np.trapezoid(unnormalized, theta_grid)

    if k in [1, 2, 5, 10, 15]:
        posteriors[k] = posterior.copy()

    mean_est.append(np.trapezoid(theta_grid * posterior, theta_grid))
    map_est.append(theta_grid[np.argmax(posterior)])

# Visualize Posteriors
fig6 = go.Figure()
for k, dist in posteriors.items():
    fig6.add_trace(go.Scatter(x=theta_grid, y=dist, name=f"Step {k}"))
fig6.add_vline(x=theta_true, line_dash="dash", line_color="black", annotation_text="True Damage")
fig6.update_layout(title="Posterior Evolution over Sensor Readings", xaxis_title="Theta", yaxis_title="Density")
fig6.show()

# Visualize Convergence
fig7 = go.Figure()
steps = np.arange(n_steps + 1)
fig7.add_trace(go.Scatter(x=steps, y=mean_est, mode='lines+markers', name="Posterior Mean"))
fig7.add_trace(go.Scatter(x=steps, y=map_est, mode='lines+markers', name="MAP Estimate"))
fig7.add_hline(y=theta_true, line_dash="dash", line_color="red", annotation_text="True Theta")
fig7.update_layout(title="Estimator Convergence", xaxis_title="Inspection Step (k)", yaxis_title="Estimated Stiffness")
fig7.show()

# Q. Gaussian Mixture Clustering as Conditional Updating

Consider a dataset
$$
x_1,x_2,\dots,x_n\in\mathbb R^d.
$$
We wish to cluster these observations into $K$ groups. Instead of assigning each point deterministically to a cluster at the beginning, we introduce a latent random variable
$$
C_i\in{1,\dots,K},
$$
where $C_i=k$ means that the observation $x_i$ belongs to cluster $k$.
Let the prior probability of cluster membership be
$$
P(C_i=k)=\phi_k,
$$
where
$$
\phi_k\ge 0,
\qquad
\sum_{k=1}^K \phi_k=1.
$$

Conditional on $C_i=k$, assume that the observation $X_i$ is generated from a multivariate Gaussian distribution:
$$
X_i\mid C_i=k
\sim
\mathscr N(\mu_k,\Sigma_k),
$$
where
$$
\mu_k\in\mathbb R^d,
\qquad
\Sigma_k\in\mathbb R^{d\times d}
$$
are the mean vector and covariance matrix of cluster $k$.

The model parameters
$$
\phi_k,\mu_k,\Sigma_k,
\qquad k=1,\dots,K,
$$
are assumed to be fixed but unknown.

---

1. Deriving the Marginal Density:
Using the law of total probability, show that the marginal density of $X_i$ is
$$
p(x_i)=\sum_{k=1}^K
\phi_k
\mathscr N(x_i\mid \mu_k,\Sigma_k).
$$
Explain why this density is called a Gaussian mixture density.

---

2. Deriving the Posterior Cluster Probability:
For a fixed observation $x_i$, use Bayes' rule to derive
$$
P(C_i=k\mid X_i=x_i)=\frac{
P(X_i=x_i\mid C_i=k)P(C_i=k)
}{
\sum_{j=1}^K P(X_i=x_i\mid C_i=j)P(C_i=j)
}.
$$
Then substitute the Gaussian model and the cluster prior to obtain
$$
P(C_i=k\mid X_i=x_i)=\frac{
\phi_k\mathscr N(x_i\mid \mu_k,\Sigma_k)
}{
\sum_{j=1}^K
\phi_j\mathscr N(x_i\mid \mu_j,\Sigma_j)
}.
$$
This quantity is called the responsibility of cluster $k$ for data point $x_i$, and is denoted by
$$
\gamma_{ik}=P(C_i=k\mid X_i=x_i).
$$
Explain why $\gamma_{ik}$ may be interpreted as a posterior probability of cluster membership.

---

3. One-Hot Encoding of the Latent Cluster Variable:
Now define a one-hot encoded latent random vector
$$
Z_i=
\begin{bmatrix}
Z_{i1}\\
Z_{i2}\\
\vdots\\
Z_{iK}
\end{bmatrix},
$$
where
$$
Z_{ik}=\begin{cases}
1, & \text{if } C_i=k,\\
0, & \text{otherwise}.
\end{cases}
$$
Show that
$$
\mathbb E[Z_{ik}\mid X_i=x_i]=P(C_i=k\mid X_i=x_i).
$$
Hence show that
$$
\mathbb E[Z_i\mid X_i=x_i]=\begin{bmatrix}
\gamma_{i1}\\
\gamma_{i2}\\
\vdots\\
\gamma_{iK}
\end{bmatrix}.
$$
Conclude that the soft cluster assignment in a Gaussian mixture model is precisely the conditional expectation
$$
\mathbb E[Z_i\mid X_i=x_i].
$$

---

4. From Soft Assignment to Hard Clustering:
The vector
$$
\mathbb E[Z_i\mid X_i=x_i]
$$
gives a soft assignment of $x_i$ to all clusters. A hard cluster assignment can be obtained by choosing the cluster with the largest posterior probability:
$$
\widehat C_i=\operatorname{arg\,max}_{1\le k\le K}
\gamma_{ik}.
$$
Explain the difference between soft clustering and hard clustering in this context.

---

5. Conditional Expectation of the Observation Given the Cluster:
Show that
$$
\mathbb E[X_i\mid C_i=k]=\mu_k.
$$
Explain why $\mu_k$ can be interpreted as the center of cluster $k$.
Then compare the two conditional expectations
$$
\mathbb E[Z_i\mid X_i=x_i]
$$
and
$$
\mathbb E[X_i\mid C_i=k].
$$
Explain why the first gives the soft cluster membership of an observed point, while the second gives the mean location of a cluster.

---

6. The Complete-Data Likelihood
If the latent labels $z_i$ were known, the complete-data likelihood would be
$$
p(x_1,\dots,x_n,z_1,\dots,z_n)=\prod_{i=1}^n
\prod_{k=1}^K
\left[
\phi_k
\mathscr N(x_i\mid \mu_k,\Sigma_k)
\right]^{z_{ik}}.
$$
Take the logarithm and show that the complete-data log-likelihood is
$$
\ell_c=\sum_{i=1}^n
\sum_{k=1}^K
z_{ik}
\left[
\log \phi_k
+
\log \mathscr N(x_i\mid \mu_k,\Sigma_k)
\right].
$$
Explain why this expression would be easy to maximize if the values of $z_{ik}$ were known.

---

7. The EM Interpretation:
In practice, the latent variables $Z_i$ are not observed. The EM algorithm replaces the unknown indicators $z_{ik}$ by their conditional expectations given the observed data and current parameter estimates:
$$
z_{ik}
\quad\leadsto\quad
\mathbb E[Z_{ik}\mid X_i=x_i].
$$
That is,
$$
z_{ik}
\quad\leadsto\quad
\gamma_{ik}.
$$
This is the E-step of the EM algorithm.
Using this idea, write the expected complete-data log-likelihood:
$$
Q=\sum_{i=1}^n
\sum_{k=1}^K
\gamma_{ik}
\left[
\log \phi_k
+
\log \mathscr N(x_i\mid \mu_k,\Sigma_k)
\right].
$$
Explain why the E-step can be interpreted as a conditional update of cluster membership probabilities.

---

8. Parameter Updates:
By maximizing $Q$ with respect to the model parameters, derive the standard GMM updates:
$$
N_k=\sum_{i=1}^n \gamma_{ik},
$$
$$
\phi_k^{\text{new}}=\frac{N_k}{n},
$$
$$
\mu_k^{\text{new}}=\frac{1}{N_k}
\sum_{i=1}^n
\gamma_{ik}x_i,
$$
and
$$
\Sigma_k^{\text{new}}=\frac{1}{N_k}
\sum_{i=1}^n
\gamma_{ik}
(x_i-\mu_k^{\text{new}})
(x_i-\mu_k^{\text{new}})^T.
$$
Explain how the responsibility $\gamma_{ik}$ acts as a fractional membership weight of observation $x_i$ in cluster $k$.

---

9. Interpretation:
Write a short paragraph explaining why GMM clustering can be viewed as a repeated process of conditional updating.
Your answer should mention the following points:

* The mixture weight $\phi_k$ is the prior probability of cluster $k$.
* The Gaussian density $\mathscr N(x_i\mid \mu_k,\Sigma_k)$ measures how compatible $x_i$ is with cluster $k$.
* The responsibility $\gamma_{ik}$ is the posterior probability of cluster $k$ after observing $x_i$.
* The soft assignment vector is
$$
\mathbb E[Z_i\mid X_i=x_i].
$$

* The M-step updates the cluster parameters using these posterior membership probabilities as weights.
Conclude that Gaussian mixture clustering is probabilistic clustering based on conditional expectations of latent cluster membership variables.

---

Here is a perfectly tailored question that you can add as the final part (**Part 10**) of your assignment notebook to bridge your theoretical derivations with your code implementation:

---

10. Computational Simulation and Out-of-Sample Validation

Using the theoretical framework established in the previous parts, write a Python class named `GMMFinancialSegmenter` that implements a two-dimensional Gaussian Mixture Model (GMM) using `scikit-learn` and visualizes the results interactively using `Plotly`. Your implementation should fulfill the following criteria:

* **Data Splitting and Scaling:** Accept a dataset containing two continuous features (e.g., mimicking financial behaviors like `PURCHASES` and `CREDIT_LIMIT`), standardize the features to handle variance scaling, and split the data into an 80% training set and a 20% validation/test set.
* **EM Execution:** Fit a GMM with $K=3$ components on the training data using the Expectation-Maximization (EM) algorithm, printing whether the model successfully converged and the number of iterations required.
* **Out-of-Sample Performance:** Compute and output the average log-likelihood score over the unseen test set to validate how well the learned density functions generalize to new data.
* **Interactive Visualizations:** Implement methods to generate three distinct Plotly figures:
1. An empirical **2D Density Heatmap** of the raw training data with marginal distributions to inspect its underlying multimodal structure.
2. A **Training Assignment Plot** that overlays the training data points on top of a continuous contour map showing the maximum posterior responsibilities ($\gamma_{ik}$) computed across a fine coordinate grid.
3. A **Test Assignment Plot** that replicates the contour boundary visualization but overlays out-of-sample test data points to expose the physical regions of cluster ambiguity.



Briefly evaluate the resulting plots. Explain how the continuous background contour map visually demonstrates the soft assignment expectation vector $\mathbb{E}[Z_i \mid X_i = x_{\text{grid}}]$ that you proved analytically in Part 3.

Use the dataset

https://www.kaggle.com/datasets/arjunbhasin2013/ccdata

1. By the law of total probability, $p(x_i) = \sum_{k=1}^K P(X_i=x_i \mid C_i=k) P(C_i=k)$. Substituting the given distributions, $p(x_i) = \sum_{k=1}^K \phi_k \mathscr{N}(x_i \mid \mu_k, \Sigma_k)$. This is a Gaussian mixture density because it models the overall data distribution as a weighted convex combination ("mixture") of individual Gaussian components.

2. Applying Bayes' theorem to the given priors and likelihoods yields $\gamma_{ik}$. This is interpreted as a posterior probability because it represents our updated belief about which cluster generated the point $x_i$ after observing its specific spatial features.

3. By definition of expectations for indicator variables, $\mathbb{E}[Z_{ik} \mid X_i=x_i] = 1 \cdot P(Z_{ik}=1 \mid X_i=x_i) + 0 \cdot P(Z_{ik}=0 \mid X_i=x_i) = P(C_i=k \mid X_i=x_i) = \gamma_{ik}$. Stacking these into a vector provides the soft cluster assignment, confirming it is exactly the conditional expectation $\mathbb{E}[Z_i \mid X_i=x_i]$.

4. Soft clustering assigns a probability vector to each point (e.g., 70% cluster A, 30% cluster B), capturing uncertainty. Hard clustering forces a mutually exclusive assignment by taking the argmax of the soft probabilities, collapsing the uncertainty into a single deterministic label.

5. For a Gaussian distribution, the expected value is its mean parameter, thus $\mathbb{E}[X_i \mid C_i=k] = \mu_k$, which defines the spatial center of the cluster. $\mathbb{E}[Z_i \mid X_i=x_i]$ operates in the latent discrete space (membership probabilities), while $\mathbb{E}[X_i \mid C_i=k]$ operates in the continuous feature space (spatial locations).

6. Taking the logarithm of the given complete-data likelihood converts the product into sums:
  $$\ell_c = \sum_{i=1}^n \sum_{k=1}^K z_{ik} \left[ \log \phi_k + \log \mathscr{N}(x_i \mid \mu_k, \Sigma_k) \right]$$If $z_{ik}$ were known, this simplifies into completely decoupled sums for each cluster, allowing the parameters $(\phi_k, \mu_k, \Sigma_k)$ to be maximized independently using standard closed-form MLE formulas.

  7. Substituting $z_{ik}$ with $\gamma_{ik}$ computes the expected complete-data log-likelihood ($Q$). This E-step is a conditional update because it calculates the current best guess of membership probabilities based conditionally on the current spatial parameter estimates.

  8. Maximizing $Q$ yields formulas where $\gamma_{ik}$ acts as a fractional weight. Instead of a point $x_i$ fully belonging to a single cluster, it contributes a fraction ($\gamma_{ik}$) of its value to cluster $k$'s mean and covariance calculations

  9. Gaussian Mixture Model clustering is fundamentally a repeated process of conditional updating. The mixture weight $\phi_k$ acts as the initial prior probability of a point belonging to cluster $k$. We then measure how compatible a specific observation $x_i$ is with cluster $k$ using the Gaussian density $\mathscr{N}(x_i \mid \mu_k, \Sigma_k)$. Combining these via Bayes' rule gives the responsibility $\gamma_{ik}$, representing the updated posterior probability of cluster $k$ after observation. These responsibilities form the soft assignment vector $\mathbb{E}[Z_i \mid X_i=x_i]$. Finally, the M-step performs a conditional update on the cluster spatial parameters using these posterior probabilities as fractional weights. Consequently, GMM is a fully probabilistic clustering paradigm built natively upon conditional expectations of latent variables

In [3]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

class GMMFinancialSegmenter:
    def __init__(self, n_components=3):
        self.gmm = GaussianMixture(n_components=n_components, covariance_type='full', random_state=42)
        self.scaler = StandardScaler()

    def prepare_data(self, df, features):
        X = df[features].dropna().values
        X_scaled = self.scaler.fit_transform(X)
        self.X_train, self.X_test = train_test_split(X_scaled, test_size=0.2, random_state=42)
        return self.X_train, self.X_test

    def fit_and_evaluate(self):
        self.gmm.fit(self.X_train)
        print(f"Model Converged: {self.gmm.converged_}")
        print(f"Iterations: {self.gmm.n_iter_}")

        test_score = self.gmm.score(self.X_test)
        print(f"Out-of-Sample Average Log-Likelihood: {test_score:.4f}")

    def plot_heatmap(self):
        fig = px.density_heatmap(x=self.X_train[:, 0], y=self.X_train[:, 1],
                                 marginal_x="histogram", marginal_y="histogram",
                                 title="2D Density Heatmap of Training Data")
        fig.show()

    def plot_assignments(self, data, title):
        # Create a mesh grid
        x_min, x_max = data[:, 0].min() - 1, data[:, 0].max() + 1
        y_min, y_max = data[:, 1].min() - 1, data[:, 1].max() + 1
        xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100), np.linspace(y_min, y_max, 100))

        # Predict responsibilities on grid (Hard assignment for background)
        Z = self.gmm.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

        fig = go.Figure()
        fig.add_trace(go.Contour(x=np.linspace(x_min, x_max, 100),
                                 y=np.linspace(y_min, y_max, 100),
                                 z=Z, colorscale='Viridis', opacity=0.3, showscale=False))

        preds = self.gmm.predict(data)
        fig.add_trace(go.Scatter(x=data[:, 0], y=data[:, 1], mode='markers',
                                 marker=dict(color=preds, colorscale='Viridis', line_width=1)))
        fig.update_layout(title=title, xaxis_title="Feature 1 (Scaled)", yaxis_title="Feature 2 (Scaled)")
        fig.show()

# --- Execution Example ---
# NOTE: Replace with your actual downloaded Kaggle dataset path
# df = pd.read_csv('CC GENERAL.csv')
# df['CREDIT_LIMIT'] = df['CREDIT_LIMIT'].fillna(df['CREDIT_LIMIT'].median())

# Mock data to demonstrate class functionality natively in Colab
np.random.seed(42)
mock_data = pd.DataFrame({
    'PURCHASES': np.concatenate([np.random.normal(500, 200, 300), np.random.normal(3000, 500, 300)]),
    'CREDIT_LIMIT': np.concatenate([np.random.normal(1500, 400, 300), np.random.normal(6000, 1000, 300)])
})

segmenter = GMMFinancialSegmenter(n_components=3)
X_train, X_test = segmenter.prepare_data(mock_data, ['PURCHASES', 'CREDIT_LIMIT'])
segmenter.fit_and_evaluate()
segmenter.plot_heatmap()
segmenter.plot_assignments(X_train, "Training Assignment Plot")
segmenter.plot_assignments(X_test, "Test Assignment Plot")

Model Converged: True
Iterations: 5
Out-of-Sample Average Log-Likelihood: -0.7082
